In [1]:
from IPython.display import display, HTML
display(HTML('<style>.container { width:80% !important; }</style>'))

In [2]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
import math
import urllib.parse
import json
import uuid

In [3]:
pd.set_option('display.max_columns', 500) # To utilise larger part of screen

pd.set_option('display.max_colwidth', None) # To show full cell text

In [4]:
data_path = 'data/cb_20240203'

# Data Import

In [5]:
organisations = pd.read_csv(data_path + '/organizations.csv')[['uuid', 'name', 'country_code', 'region', 'city', 'status', 'short_description', 'category_list', 'category_groups_list', 'num_funding_rounds', 'total_funding_usd', 'founded_on', 'logo_url']]
organisations = organisations.rename(columns = {'uuid': 'org_uuid', 'name': 'org_name', 'country_code': 'org_country_code', 'region': 'org_region', 'city': 'org_city', 'logo_url': 'org_logo_url'})

organisations = organisations.assign(founded_on = pd.to_datetime(organisations['founded_on'], errors = 'coerce'))
organisations['total_funding_usd'] = organisations['total_funding_usd'].astype('Int64')
organisations['num_funding_rounds'] = organisations['num_funding_rounds'].astype('Int64')

organisations.head()

,org_uuid,org_name,org_country_code,org_region,org_city,status,short_description,category_list,category_groups_list,num_funding_rounds,total_funding_usd,founded_on,org_logo_url
0,e1393508-30ea-8a36-3f96-dd3226033abd,Wetpaint,USA,New York,New York,acquired,Wetpaint offers an online social publishing platform that helps digital publishers grow their customer base.,"Publishing,Social Media,Social Media Management","Content and Publishing,Internet Services,Media and Entertainment,Sales and Marketing",3,39750000,2005-06-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180177/2036b3394a37152e0ff69f27c71bc883.jpg
1,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Zoho,USA,California,Pleasanton,operating,"Zoho offers a suite of business, collaboration, and productivity applications.","Cloud Computing,Collaboration,Developer Tools,Enterprise Software,Information Services,Information Technology,Network Security,Project Management,Software,Web Apps","Administrative Services,Apps,Information Technology,Internet Services,Other,Privacy and Security,Software",<NA>,<NA>,1996-03-17,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180181/f8aaab73f17af0296eba5deda7a5b95b.png
2,5f2b40b8-d1b3-d323-d81a-b7a8e89553d0,Digg,USA,New York,New York,acquired,"Digg Inc. operates a website that enables its users to find, read, and share the most interesting and talked about stories on the internet.","Internet,Social Media,Social Network","Internet Services,Media and Entertainment",6,49000000,2004-10-11,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180182/e77f8f561153ffb45a9ffd538978380d.jpg
3,f4d5ab44-058b-298b-ea81-380e6e9a8eec,Omidyar Network,USA,California,Redwood City,operating,Omidyar Network is an investment firm.,"Enterprise Software,Financial Services,Venture Capital","Financial Services,Lending and Investments,Software",<NA>,<NA>,2004-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1491524876/tjg7440r2uc0s5neswkp.png
4,df662812-7f97-0b43-9d3e-12f64f504fbb,Meta,USA,California,Menlo Park,ipo,"Meta is a social technology company that enables people to connect, find communities, and grow businesses.","Augmented Reality,Metaverse,Mortgage,Social Media,Social Network,Virtual Reality","Hardware,Internet Services,Media and Entertainment,Real Estate,Software",14,24607817488,2004-02-04,https://images.crunchbase.com/image/upload/t_cb-default-original/whm4ed1rrc8skbdi3biv


In [6]:
people = pd.read_csv(data_path + '/people.csv')[['uuid', 'name', 'country_code', 'region', 'city', 'featured_job_organization_uuid', 'featured_job_title', 'logo_url']]
people = people.rename(columns = {'uuid': 'person_uuid', 'name': 'person_name', 'country_code': 'person_country_code', 'region': 'person_region', 'city': 'person_city', 'featured_job_organization_uuid': 'featured_job_org_uuid', 'logo_url': 'person_logo_url'})

people.head()

,person_uuid,person_name,person_country_code,person_region,person_city,featured_job_org_uuid,featured_job_title,person_logo_url
0,ed13cd36-fe2b-3707-197b-0c2d56e37a71,Ben Elowitz,USA,Washington,Seattle,1d845b32-7d80-47af-957d-78ccbeeaefb6,Co-Founder,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180224/56303d2f4b99dd1dcc8abf17ba3fd8bf.jpg
1,5ceca97b-493c-1446-6249-5aaa33464763,Kevin Flaherty,USA,Washington,Mercer Island,789e5e4d-0c90-d06e-92a0-b800b461c3da,Team Member,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180225/a307ba697e0f042623f05f35477bf495.jpg
2,9f99a98a-aa97-b30b-0d36-db67c1d277e0,Raju Vegesna,USA,California,San Francisco,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Chief Evangelist,https://images.crunchbase.com/image/upload/t_cb-default-original/oaxfoww3m0u0lwdovzv3
3,6e1bca72-a865-b518-b305-31214ce2d1b0,Ian Wenig,NaN,NaN,NaN,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,VP Business Development,https://images.crunchbase.com/image/upload/t_cb-default-original/v1442309935/yiajbpfbxyopc5l4zs4t.png
4,3b598c59-7b6c-2d48-763c-da55bca77035,Owen Byrne,USA,California,Mountain View,NaN,NaN,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180229/eb6f898475c0bef4c624ed44c9add19a.jpg


In [7]:
jobs = pd.read_csv(data_path + '/jobs.csv')[['uuid', 'person_uuid', 'org_uuid', 'started_on', 'ended_on', 'is_current', 'title', 'job_type']]
jobs = jobs.rename(columns = {'uuid': 'job_uuid', 'title': 'job_title'})

jobs = jobs.assign(started_on = pd.to_datetime(jobs['started_on'], errors = 'coerce'))
jobs = jobs.assign(ended_on = pd.to_datetime(jobs['ended_on'], errors = 'coerce'))

jobs.head()

,job_uuid,person_uuid,org_uuid,started_on,ended_on,is_current,job_title,job_type
0,697b6934-fc1f-9d63-cfb2-1a10759b378e,ed13cd36-fe2b-3707-197b-0c2d56e37a71,e1393508-30ea-8a36-3f96-dd3226033abd,2005-10-01,2014-06-01,False,Co-Founder and CEO,executive
1,b1de3765-442e-b556-9304-551c2a055901,5ceca97b-493c-1446-6249-5aaa33464763,e1393508-30ea-8a36-3f96-dd3226033abd,NaT,NaT,False,VP Marketing,executive
2,1319cd30-f5e8-c700-0af6-64029c6f7124,9f99a98a-aa97-b30b-0d36-db67c1d277e0,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,2000-11-01,NaT,True,Chief Evangelist,employee
3,27a252de-1ea8-c620-b2d4-5b889fa9b40f,6e1bca72-a865-b518-b305-31214ce2d1b0,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,2006-03-01,NaT,True,VP Business Development,executive
4,5a802a79-229f-44ae-0aba-db330f10b67a,c92a1f00-8c19-bf2e-0f28-dbbd383dc968,5f2b40b8-d1b3-d323-d81a-b7a8e89553d0,2005-07-01,2010-04-05,False,CEO,executive


In [8]:
funding_rounds = pd.read_csv(data_path + '/funding_rounds.csv')[['uuid', 'investment_type', 'announced_on', 'raised_amount_usd', 'org_uuid']]

funding_rounds = funding_rounds.rename(columns = {'uuid': 'funding_round_uuid'})

funding_rounds = funding_rounds.assign(announced_on = pd.to_datetime(funding_rounds['announced_on'], errors = 'coerce'))

funding_rounds.head()

,funding_round_uuid,investment_type,announced_on,raised_amount_usd,org_uuid
0,8a945939-18e0-cc9d-27b9-bf33817b2818,angel,2004-09-01,500000.0,df662812-7f97-0b43-9d3e-12f64f504fbb
1,d950d7a5-79ff-fb93-ca87-13386b0e2feb,series_a,2005-05-01,12700000.0,df662812-7f97-0b43-9d3e-12f64f504fbb
2,6fae3958-a001-27c0-fb7e-666266aedd78,series_b,2006-04-01,27500000.0,df662812-7f97-0b43-9d3e-12f64f504fbb
3,bcd5a63d-ed99-6963-0dd2-e36f6582f846,series_b,2006-05-01,10500000.0,f53cb4de-236e-0b1b-dee8-7104a8b018f9
4,60e6afd9-1215-465a-dd17-0ed600d4e29b,series_a,2007-01-17,NaN,4111dc8b-c0df-2d24-ed33-30cd137b3098


In [9]:
investments = pd.read_csv(data_path + '/investments.csv')[['uuid', 'funding_round_uuid', 'investor_uuid']]

investments = investments.rename(columns = {'uuid': 'investment_uuid'})

investments.head()

,investment_uuid,funding_round_uuid,investor_uuid
0,524986f0-3049-54a4-fa72-f60897a5e61d,d950d7a5-79ff-fb93-ca87-13386b0e2feb,b08efc27-da40-505a-6f9d-c9e14247bf36
1,6556ab92-6465-25aa-1ffc-7f8b4b09a476,6fae3958-a001-27c0-fb7e-666266aedd78,e2006571-6b7a-e477-002a-f7014f48a7e3
2,0216e06a-61f8-9cf1-19ba-20811229c53e,6fae3958-a001-27c0-fb7e-666266aedd78,8d5c7e48-82da-3025-dd46-346a31bab86f
3,dadd7d86-520d-5e35-3033-fc1d8792ab91,bcd5a63d-ed99-6963-0dd2-e36f6582f846,7ca12f7a-2f8e-48b4-a8d1-1a33a0e275b9
4,581c4b38-9653-7117-9bd4-7ffe5c7eba69,60e6afd9-1215-465a-dd17-0ed600d4e29b,fb2f8884-ec07-895a-48d7-d9a9d4d7175c


In [10]:
investment_partners = pd.read_csv(data_path + '/investment_partners.csv')[['uuid', 'funding_round_uuid', 'partner_uuid']]

investment_partners = investment_partners.rename(columns = {'uuid': 'investment_partner_uuid'})

investment_partners.head()

,investment_partner_uuid,funding_round_uuid,partner_uuid
0,524986f0-3049-54a4-fa72-f60897a5e61d,d950d7a5-79ff-fb93-ca87-13386b0e2feb,2d78d1e7-203c-3eb6-bf1b-c51f10e0679b
1,524986f0-3049-54a4-fa72-f60897a5e61d,d950d7a5-79ff-fb93-ca87-13386b0e2feb,eaf6c243-d355-32f3-e23a-2a5fc82e8b34
2,6556ab92-6465-25aa-1ffc-7f8b4b09a476,6fae3958-a001-27c0-fb7e-666266aedd78,478e7efd-bec4-b9f5-304b-cffedc1fc012
3,0216e06a-61f8-9cf1-19ba-20811229c53e,6fae3958-a001-27c0-fb7e-666266aedd78,0f9f3c05-cb79-f58f-6cc5-98ddc8382d4f
4,dadd7d86-520d-5e35-3033-fc1d8792ab91,bcd5a63d-ed99-6963-0dd2-e36f6582f846,ea9f4980-600c-84f4-a5d6-b4f8c2f787fb


In [11]:
acquisitions = pd.read_csv(data_path + '/acquisitions.csv')[['acquiree_uuid', 'acquirer_uuid', 'type', 'acquirer_name', 'acquired_on', 'acquisition_type', 'price_usd']]

acquisitions = acquisitions.rename(columns = {'acquiree_uuid': 'org_uuid', 'type': 'exit_type', 'acquired_on': 'exit_date', 'price_usd': 'exit_valuation'})

acquisitions = acquisitions.assign(exit_date = pd.to_datetime(acquisitions['exit_date'], errors = 'coerce'))

acquisitions['exit_valuation'] = acquisitions['exit_valuation'].astype('Int64')

acquisitions.head()

,org_uuid,acquirer_uuid,exit_type,acquirer_name,exit_date,acquisition_type,exit_valuation
0,180ebf67-68d0-2316-e93d-8e1e546330ba,d70777cc-14bd-2416-0692-5a483781b78b,acquisition,Fox Interactive Media,2007-05-30,NaN,<NA>
1,5b05e013-a448-3a0b-d872-a6ae668e1192,6acfa7da-1dbd-936e-d985-cf07a1b27711,acquisition,Google,2007-07-01,NaN,60000000
2,8249dffa-1ca6-6f99-9f76-d56c83f85f2d,f09c1228-2e7d-1889-6647-ba5021b2e4ea,acquisition,CBS Entertainment,2007-05-01,NaN,280000000
3,10dd03fa-69ff-3a82-6321-c6b16c9a9f41,6acfa7da-1dbd-936e-d985-cf07a1b27711,acquisition,Google,2007-05-23,acquisition,100000000
4,0af10345-613d-e144-f8bd-b62e288985a0,b5a96cd7-044d-70f0-04c5-f125e57a4b35,acquisition,Scripps Networks,2007-07-01,NaN,<NA>


In [12]:
ipos = pd.read_csv(data_path + '/ipos.csv')[['org_uuid', 'type', 'went_public_on', 'valuation_price_usd']]

ipos = ipos.rename(columns = {'uuid': 'investment_partner_uuid', 'type': 'exit_type', 'went_public_on': 'exit_date', 'valuation_price_usd': 'exit_valuation'})

ipos = ipos.assign(exit_date = pd.to_datetime(ipos['exit_date'], errors = 'coerce'))

ipos['exit_valuation'] = ipos['exit_valuation'].astype('Int64')

ipos.head()

,org_uuid,exit_type,exit_date,exit_valuation
0,fd80725f-53fc-7009-9878-aeecf1e9ffbb,ipo,1986-03-13,<NA>
1,756936c0-c335-f0ae-0a3d-fe26bdff5695,ipo,1978-01-13,<NA>
2,73296f0d-85a5-78d5-90b3-86c5f8981ba9,ipo,2006-10-22,160000000
3,ff8439cf-097c-a88a-9bb9-dd83d23aa14b,ipo,1999-12-02,<NA>
4,ab8e5ba4-df5d-121b-93b6-eae7a0c89245,ipo,1988-08-12,6000000000


In [13]:
acquisitons_ipos = pd.concat([acquisitions, ipos.drop_duplicates(subset = ['org_uuid', 'exit_date'])], ignore_index = True).sort_values(by = 'exit_date', ascending = False).drop_duplicates(subset = ['org_uuid'])

acquisitons_ipos.head()

,org_uuid,acquirer_uuid,exit_type,acquirer_name,exit_date,acquisition_type,exit_valuation
161080,3b46aeb5-9e8b-96ae-eaa0-a7ac254987d9,d53020f6-8a56-9b9a-764d-bf209256b5b3,acquisition,BAE Systems,2024-02-02,acquisition,<NA>
161066,8512ea38-eeec-46c3-a540-020a7e3bd1ae,4bbe63fc-fccb-4b52-981c-5ac694ca63a7,acquisition,Lyvia Group,2024-02-02,acquisition,<NA>
161064,47b41a35-6a1e-494a-98cc-a04692321365,b19fb1ee-c956-8fe9-284b-ce0b1aa02929,acquisition,Indutrade,2024-02-02,acquisition,<NA>
211096,14a8cd39-010e-ea26-8afd-513320dd9e3c,NaN,ipo,NaN,2024-02-02,NaN,<NA>
161059,4af6fb74-d81b-4d9d-8dab-9317d0ab13af,a621163d-0701-e38d-a9e6-d7778da72eac,acquisition,Bounteous,2024-02-01,merge,<NA>


# Defining Constants

In [14]:
# Years after foundation to consider
years_founding_cutoff = 3

# Define the priority list
relation_priority = ['executive', 'board_member', 'advisor', 'investor']

# Create a mapping for priority
relation_priority_map = {value: index for index, value in enumerate(relation_priority)}

# Creating Main Dataset

## alumni_master - Information about everyone's jobs and subsequent jobs/investments

In [15]:
investments_info = pd.merge(
    investments,
    funding_rounds,
    how = 'left',
    on = 'funding_round_uuid'
).rename(columns = {'investor_uuid': 'person_uuid', 'announced_on': 'started_on'})[['person_uuid', 'org_uuid', 'started_on']]

investments_info = investments_info.assign(job_type = 'investor')
investments_info = investments_info.assign(job_title = 'investor')

investments_info

,person_uuid,org_uuid,started_on,job_type,job_title
0,b08efc27-da40-505a-6f9d-c9e14247bf36,df662812-7f97-0b43-9d3e-12f64f504fbb,2005-05-01,investor,investor
1,e2006571-6b7a-e477-002a-f7014f48a7e3,df662812-7f97-0b43-9d3e-12f64f504fbb,2006-04-01,investor,investor
2,8d5c7e48-82da-3025-dd46-346a31bab86f,df662812-7f97-0b43-9d3e-12f64f504fbb,2006-04-01,investor,investor
3,7ca12f7a-2f8e-48b4-a8d1-1a33a0e275b9,f53cb4de-236e-0b1b-dee8-7104a8b018f9,2006-05-01,investor,investor
4,fb2f8884-ec07-895a-48d7-d9a9d4d7175c,4111dc8b-c0df-2d24-ed33-30cd137b3098,2007-01-17,investor,investor
...,...,...,...,...,...
1014561,bbbc8882-7d90-46ec-adef-1006be759b07,8293c5da-8121-46f1-bc95-d08a1ce1f7e1,2024-01-25,investor,investor
1014562,8e6b35ad-3a52-45b6-ac7f-150d756e0d9c,8293c5da-8121-46f1-bc95-d08a1ce1f7e1,2024-01-25,investor,investor
1014563,e758f155-0c21-4a3e-b972-8cd511f3b86b,f4345555-cad7-55dc-6c0b-55c8f3318121,2017-09-01,investor,investor
1014564,e758f155-0c21-4a3e-b972-8cd511f3b86b,14d98753-5f89-4f6c-a964-f6de2c28d4c1,2019-04-01,investor,investor


In [16]:
jobs_in_scope = jobs.loc[jobs['job_type'].isin(['executive', 'investor', 'advisor', 'board_member'])][['person_uuid', 'org_uuid', 'started_on', 'ended_on', 'job_title', 'job_type']]

jobs_investments = pd.concat([jobs_in_scope, investments_info.loc[investments_info['person_uuid'].isin(jobs_in_scope['person_uuid'])]], ignore_index = True)

jobs_investments = pd.merge(jobs_investments, organisations[['org_uuid', 'org_name', 'org_country_code', 'org_city', 'founded_on', 'short_description', 'total_funding_usd', 'org_logo_url']], how = 'left', on = 'org_uuid')

jobs_investments = pd.merge(jobs_investments, people[['person_uuid', 'person_name', 'person_logo_url']], how = 'left', on = 'person_uuid')

all_info = pd.merge(jobs_investments, acquisitons_ipos, how = 'left', on = 'org_uuid')

subsequent_exploded = pd.merge(all_info, all_info, on = 'person_uuid', suffixes = ('_target', '_subsequent'))

alumni_master = subsequent_exploded.loc[
    (subsequent_exploded['org_uuid_target'] != subsequent_exploded['org_uuid_subsequent']) # Remove entry of target org
    & (subsequent_exploded['job_type_target'] == 'executive') # Target org role must be executive
    & (subsequent_exploded['started_on_subsequent'] >= subsequent_exploded['started_on_target']) # Subsequrnt must be after target start date
    & (subsequent_exploded['started_on_target'] <= subsequent_exploded['founded_on_target'] + pd.DateOffset(years = years_founding_cutoff))
].rename(columns = {'person_name_target': 'person_name', 'job_type_subsequent': 'relation_type', 'person_logo_url_target': 'person_logo_url'}).drop(columns = ['person_name_subsequent', 'person_logo_url_subsequent'], axis = 1)

alumni_master = alumni_master.drop(columns = ['acquirer_uuid_target', 'exit_type_target', 'acquirer_name_target', 'exit_date_target', 'acquisition_type_target', 'exit_valuation_target', 'short_description_target', 'total_funding_usd_target', 'org_logo_url_target'], axis = 1)

# Prioritise which relation with same subsequent company should be kep
alumni_master['priority'] = alumni_master['relation_type'].map(relation_priority_map)
alumni_master = alumni_master.sort_values(by = 'priority')

# Drop the priority column as it's no longer needed
alumni_master = alumni_master.drop(columns = ['priority'], axis = 1)

# Define columns to exclude
exclude_columns = ['job_title_subsequent', 'relation_type', 'started_on_subsequent', 'ended_on_subsequent']

# Get all columns except for the ones to exclude
subset_columns = alumni_master.columns.difference(exclude_columns).tolist()

# Drop duplicates keeping the one with the highest priority
alumni_master = alumni_master.drop_duplicates(subset = subset_columns, keep = 'first')

alumni_master = alumni_master.dropna(subset = ['person_uuid', 'org_uuid_target', 'org_uuid_subsequent'])

alumni_master['record_uuid'] = [uuid.uuid4() for _ in range(len(alumni_master))] # Adding record_uuid as primary key in DB

alumni_master

,person_uuid,org_uuid_target,started_on_target,ended_on_target,job_title_target,job_type_target,org_name_target,org_country_code_target,org_city_target,founded_on_target,person_name,person_logo_url,org_uuid_subsequent,started_on_subsequent,ended_on_subsequent,job_title_subsequent,relation_type,org_name_subsequent,org_country_code_subsequent,org_city_subsequent,founded_on_subsequent,short_description_subsequent,total_funding_usd_subsequent,org_logo_url_subsequent,acquirer_uuid_subsequent,exit_type_subsequent,acquirer_name_subsequent,exit_date_subsequent,acquisition_type_subsequent,exit_valuation_subsequent,record_uuid
3,ed13cd36-fe2b-3707-197b-0c2d56e37a71,e1393508-30ea-8a36-3f96-dd3226033abd,2005-10-01,2014-06-01,Co-Founder and CEO,executive,Wetpaint,USA,New York,2005-06-01,Ben Elowitz,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180224/56303d2f4b99dd1dcc8abf17ba3fd8bf.jpg,cf253887-5eac-21a2-28d3-47db7311f7e9,2018-06-01,2019-03-01,Managing Director,executive,Madrona,USA,Seattle,1995-01-01,Madrona is a venture firm that invests in early- to late-stage companies in the Pacific Northwest and beyond.,<NA>,https://images.crunchbase.com/image/upload/t_cb-default-original/cpqilcg3lmakawavpmiv,NaN,NaN,NaN,NaT,NaN,<NA>,3b3ba8b2-fee9-499c-9205-b7dc6d486b99
6581124,ce1f1a84-0adc-72c0-661c-804b49f09830,7fb0c91a-f27d-51dc-2b58-f939265cc443,2007-03-01,2016-01-01,President/CEO,executive,First Sense Medical,USA,Pontiac,2008-01-01,Paul G. Angott,https://images.crunchbase.com/image/upload/t_cb-default-original/v1439631574/xppaezftdrbnil7orec9.png,e0f446a3-3a5d-4972-855d-5f5ef45071a6,2017-01-01,NaT,"Board of Director, McLaren Macomb Foundation",executive,McLaren Health Care,USA,Grand Blanc,1914-01-01,McLaren Health Care is a fully integrated health network committed to quality evidence-based patient care and cost efficiency.,<NA>,https://images.crunchbase.com/image/upload/t_cb-default-original/qswsltm4dqg4wwfo9siw,628008a3-3844-4d02-9fdb-f1a554616e32,acquisition,Hart Medical Equipment,2021-01-01,acquisition,<NA>,40e78654-c174-47f5-b508-9816976d7266
6581123,ce1f1a84-0adc-72c0-661c-804b49f09830,7fb0c91a-f27d-51dc-2b58-f939265cc443,2007-03-01,2016-01-01,President/CEO,executive,First Sense Medical,USA,Pontiac,2008-01-01,Paul G. Angott,https://images.crunchbase.com/image/upload/t_cb-default-original/v1439631574/xppaezftdrbnil7orec9.png,eaf1bde7-77e4-4576-9b3a-8c1f346dc266,2016-01-01,NaT,President,executive,Innovative Billboards,USA,Bloomfield Hills,2016-01-01,Innovative Billboards is a Michigan-based business .,<NA>,https://images.crunchbase.com/image/upload/t_cb-default-original/zpqhafx3w0bd8bjacjvo,NaN,NaN,NaN,NaT,NaN,<NA>,50c0f8f2-bdb9-49f6-a44e-828679181c4f
6581112,a9298caa-a2d4-f1f0-733c-cb3b12b1f9ae,11af1254-4fcc-c012-4c94-78a35487a9b7,2012-12-01,NaT,Director of Business Development,executive,Manage,USA,San Francisco,2011-01-01,Charlie Faulkner,https://images.crunchbase.com/image/upload/t_cb-default-original/v1436372022/mmwydap71ueir3djfjoc.jpg,adafd031-042c-448f-8868-97b18483c099,2020-01-01,NaT,Founder & CEO,executive,EdgeMode,USA,Chicago,2020-01-01,EdgeMode is an infrastructure management platform.,2443220,https://images.crunchbase.com/image/upload/t_cb-default-original/icxrenicsbc9vw2lnx5g,7a13ac0e-fe24-4cf0-8272-d581e66cbe7f,acquisition,Fourth Wave Energy,2021-12-06,acquisition,<NA>,c39a662b-8344-4bcc-8749-5091abc7a64e
6580877,8229962e-1865-e570-f4b4-ac6b000565b1,450635cf-96d2-79b6-df45-2db637da3ff3,2010-11-01,NaT,Co-Owner,executive,Cadenalia,ESP,Alicante,2011-01-01,César Mariel,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397186500/e155bbc6323cce94f0179a7d742f2f94.jpg,c74b4822-04d4-a319-24a1-2312bdb10c4f,2011-11-01,NaT,CEO & Founder,executive,Iristrace,NLD,The Hague,2010-01-01,"Mobile business controls, checklists and smart bots to simplify your processes, boost productivity and get the data to work for you.",1088794,https://images.crunchbase.com/image/upload/t_cb-default-original/v1436526253/

# Support functions

In [17]:
def export(df, org_uuid, name):
    df.to_json('data/output/' + name + '_' + org_uuid + '.json', orient = 'records')

# Dataframe functions

## target_org - Information about the target organisation

In [18]:
def func_target_org(org_uuid):
    target_org = organisations.loc[organisations['org_uuid'] == org_uuid][['org_uuid', 'org_name', 'org_country_code', 'org_region', 'org_city', 'short_description', 'total_funding_usd', 'founded_on', 'org_logo_url']]

    # Add exit info
    target_org = pd.merge(target_org, acquisitons_ipos, how = 'left', on = 'org_uuid')

    result = target_org.rename(columns={
        'org_uuid': 'orgUuid',
        'org_name': 'orgName',
        'org_country_code': 'orgCountryCode',
        'org_region': 'orgRegion',
        'org_city': 'orgCity',
        'short_description': 'shortDescription',
        'total_funding_usd': 'totalFundingUsd',
        'founded_on': 'foundedOn',
        'org_logo_url': 'orgLogoUrl',
        'exit_type': 'exitType',
        'exit_date': 'exitDate',
        'exit_valuation': 'exitValuation',
        'acquirer_uuid': 'acquirerUuid',
        'acquirer_name': 'acquirerName',
        'acquisition_type': 'acquisitionType',
    }).drop_duplicates()
    
    return result

## alumni_info - Information about target organisation alumnis

In [19]:
def func_alumni_info(org_uuid):
    alumni_info = alumni_master.loc[alumni_master['org_uuid_target'] == org_uuid][['person_uuid', 'person_name', 'job_title_target', 'job_type_target', 'started_on_target', 'ended_on_target', 'person_logo_url']]
    
    result = alumni_info.drop_duplicates(subset = ['person_uuid']).rename(columns={
        'person_uuid': 'personUuid',
        'person_name': 'personName',
        'job_title_target': 'jobTitle',
        'job_type_target': 'jobType',
        'started_on_target': 'startedOn',
        'ended_on_target': 'endedOn',
        'person_logo_url': 'personLogoUrl',
    }).drop_duplicates()
    
    return result

## subsequent_orgs - Organisations the alumni of the target organisation are associated with

In [20]:
def func_subsequent_orgs_info(org_uuid):
    subsequent_orgs_info = alumni_master.loc[alumni_master['org_uuid_target'] == org_uuid][['org_uuid_subsequent', 'org_name_subsequent', 'org_country_code_subsequent', 'org_city_subsequent', 'short_description_subsequent', 'total_funding_usd_subsequent', 'founded_on_subsequent', 'org_logo_url_subsequent', 'exit_type_subsequent', 'exit_date_subsequent', 'exit_valuation_subsequent', 'acquirer_uuid_subsequent', 'acquirer_name_subsequent', 'acquisition_type_subsequent']]
    
    result = subsequent_orgs_info.rename(columns={
        'org_uuid_subsequent': 'orgUuid',
        'org_name_subsequent': 'orgName',
        'org_country_code_subsequent': 'orgCountryCode',
        'org_city_subsequent': 'orgCity',
        'short_description_subsequent': 'shortDescription',
        'total_funding_usd_subsequent': 'totalFundingUsd',
        'founded_on_subsequent': 'foundedOn',
        'org_logo_url_subsequent': 'orgLogoUrl',
        'exit_type_subsequent': 'exitType',
        'exit_date_subsequent': 'exitDate',
        'exit_valuation_subsequent': 'exitValuation',
        'acquirer_uuid_subsequent': 'acquirerUuid',
        'acquirer_name_subsequent': 'acquirerName',
        'acquisition_type_subsequent': 'acquisitionType',
    }).drop_duplicates()
    
    return result

## relations - Relationships between target org people and subsequent org

In [21]:
def func_relations(org_uuid):
    relations = alumni_master.loc[alumni_master['org_uuid_target'] == org_uuid][['person_uuid', 'org_uuid_subsequent', 'relation_type', 'job_title_subsequent', 'org_logo_url_subsequent']]
    
    result = relations.rename(columns={
        'person_uuid': 'personUuid',
        'org_uuid_subsequent': 'orgUuid',
        'relation_type': 'relationType',
        'job_title_subsequent': 'jobTitleSubsequent',
        'org_logo_url_subsequent': 'orgLogoUrl',
    }).drop_duplicates()
    
    return result

# Export

In [22]:
subsequent_orgs_count = (
    alumni_master
        .groupby('org_uuid_target', as_index = False)
        .agg(
            {
                'org_uuid_subsequent': 'nunique'
            },
            skipna = True
        )
).sort_values(by = 'org_uuid_subsequent', ascending = False)

subsequent_orgs_count = pd.merge(subsequent_orgs_count, organisations, how = 'left', left_on = 'org_uuid_target', right_on = 'org_uuid')[['org_uuid_target', 'org_name', 'org_uuid_subsequent', 'org_country_code']]

subsequent_orgs_count.head(20)

,org_uuid_target,org_name,org_uuid_subsequent,org_country_code
0,2c23e7ad-4441-5320-fdf9-5aa7007fec0b,PwC,720,GBR
1,ca51c248-b907-b20b-bd0c-0d0cbd07f615,GovPredict,438,USA
2,df662812-7f97-0b43-9d3e-12f64f504fbb,Meta,388,USA
3,1bcd5c8d-5534-4f7c-8702-72d90ca9a0cb,Atom Finance,379,USA
4,96ab87ca-00b5-2ebc-f218-86262954e320,PayPal,351,USA
5,cc68526b-b2d7-4f7f-cfa7-d93b23716027,Netscape,315,USA
6,a4af56b9-4f9c-4f0e-ae9e-4d1dffe72d44,Pareto Holdings,307,USA
7,643d60b4-bfa8-ee61-3316-5af0d8325f33,Excite,290,USA
8,a7c69d60-5083-08ef-937b-6cef93a9728c,Bank of America,287,USA
9,2a56ac7e-e400-46b3-b127-9aa6c36d8d4b,Goody,286,USA


In [23]:
subsequent_orgs_count.loc[subsequent_orgs_count['org_country_code'] == 'DEU'].head(20)

,org_uuid_target,org_name,org_uuid_subsequent,org_country_code
319,31fc3998-a1f1-2e5b-6358-f8d068fa9f71,Delivery Hero,60,DEU
532,6f7ea63a-f820-733a-702b-82e75d0ac15f,Wimdu,45,DEU
540,70756e51-3859-a1ae-8edb-d7eeaf5ed342,XING,45,DEU
598,ffd3825c-ac83-fa1e-0d5b-538c08904cb6,Wunderlist,43,DEU
613,04f52814-77e2-b2c9-2d9a-4299dbb62455,Rocket Internet,42,DEU
614,dd92791e-2081-c3fc-f0a1-f1e2633dadcd,dreamfab,42,DEU
615,a6f5492b-1215-6c03-205f-1efb9913f2e2,Infineon Technologies,42,DEU
644,b43f8d93-318a-676a-8e0e-5ddc32a3c03c,StudiVZ,41,DEU
667,11639150-e8c5-dd21-2573-1b2dc7cd7b83,Spread Group,41,DEU
675,988b8953-a085-8de4-babb-fae5bb895761,Foodpanda,41,DEU


In [24]:
subsequent_orgs_count.loc[subsequent_orgs_count['org_name'] == 'Delivery Hero']

,org_uuid_target,org_name,org_uuid_subsequent,org_country_code
319,31fc3998-a1f1-2e5b-6358-f8d068fa9f71,Delivery Hero,60,DEU


In [25]:
options_json = '''
[
    {"value": "96ab87ca-00b5-2ebc-f218-86262954e320", "label": "PayPal"},
    {"value": "a367b036-5952-5435-7541-ad7ee8869e24", "label": "Tesla"},
    {"value": "df662812-7f97-0b43-9d3e-12f64f504fbb", "label": "Meta"},
    {"value": "022417b5-4980-6c54-0f3c-6736bbbb1a5e", "label": "Spotify"},
    {"value": "2cc3a5de-2303-aa00-cd1a-50bd96420392", "label": "Klarna"},
    {"value": "34035c51-8f16-4836-8f02-103392479a92", "label": "Northvolt"},
    {"value": "7b74b50c-4468-b7ec-2ff9-bf3286a399c9", "label": "Kry"},
    {"value": "d39bb6dc-582d-4878-a224-005497e03766", "label": "Voi"},
    {"value": "439d3478-40fa-e6bc-9b71-f1bfa8296f52", "label": "Epidemic Sound"},
    {"value": "6093de34-5382-f34c-a207-efa92470048b", "label": "iZettle"},
    {"value": "fd80725f-53fc-7009-9878-aeecf1e9ffbb", "label": "Microsoft"},
    {"value": "6acfa7da-1dbd-936e-d985-cf07a1b27711", "label": "Google"},
    {"value": "eef9eab2-4c50-f0a3-12b8-ce721fa2cc81", "label": "YouTube"},
    {"value": "cc68526b-b2d7-4f7f-cfa7-d93b23716027", "label": "Netscape"},
    {"value": "2c23e7ad-4441-5320-fdf9-5aa7007fec0b", "label": "PwC"},
    {"value": "b3a7efa7-e3d3-2d4c-962c-b9fdb1da496f", "label": "Opsware"},
    {"value": "1eb37109-3b93-01a9-177f-fee2cb1bfcdc", "label": "Uber"},
    {"value": "9a0e860a-7743-28e2-05c0-2b08646d0fe1", "label": "Skype"},
    {"value": "988b8953-a085-8de4-babb-fae5bb895761", "label": "Foodpanda"},
    {"value": "643d60b4-bfa8-ee61-3316-5af0d8325f33", "label": "Excite"},
    {"value": "0d5171b3-68b3-37c3-cb50-8cd8ccb8930b", "label": "Accenture"},
    {"value": "4b724583-6013-30ca-88b0-35e5c6e71ad5", "label": "Epinions"},
    {"value": "297df5af-da41-a709-7df2-7e7f04e53454", "label": "Zynga"},
    {"value": "e56b0ceb-bb30-bbec-805e-d5dc7412dcb1", "label": "eBay"},
    {"value": "a6b663d3-586a-ad04-924c-1ca87763b2fb", "label": "FreeCharge"},
    {"value": "08639f0b-56fd-997f-5c9d-0ca3b5d9672b", "label": "Justin.tv"},
    {"value": "1966bb69-4bc9-1077-c181-fe3d67509160", "label": "Pobts"},
    {"value": "86da6213-5b43-6419-4047-472102ccf66f", "label": "LinkedIn"},
    {"value": "f5c477fa-6e8c-3d64-4f2d-3603e5cc3340", "label": "Salesforce"},
    {"value": "31fc3998-a1f1-2e5b-6358-f8d068fa9f71", "label": "Delivery Hero"},
    {"value": "6f7ea63a-f820-733a-702b-82e75d0ac15f", "label": "Wimdu"},
    {"value": "70756e51-3859-a1ae-8edb-d7eeaf5ed342", "label": "Xing"},
    {"value": "ffd3825c-ac83-fa1e-0d5b-538c08904cb6", "label": "Wunderlist"},
    {"value": "04f52814-77e2-b2c9-2d9a-4299dbb62455", "label": "Rocket Internet"},
    {"value": "dd92791e-2081-c3fc-f0a1-f1e2633dadcd", "label": "dreamfab"},
    {"value": "a6f5492b-1215-6c03-205f-1efb9913f2e2", "label": "Infineon Technologies"},
    {"value": "988b8953-a085-8de4-babb-fae5bb895761", "label": "Foodpanda"},
    {"value": "36027bbe-4f12-f224-f90f-2ff31c0294af", "label": "Team Global"},
    {"value": "0a9eae4d-a269-646c-0cad-cf5b559b74b6", "label": "Zalando"},
    {"value": "251270ba-b3b8-6135-ed82-6657e1c8b046", "label": "N26"}
]
'''

options = json.loads(options_json)

## Individual files export

In [28]:
for item in options:
    target_org_uuid = item['value']
    print(target_org_uuid)
    export(func_target_org(target_org_uuid), target_org_uuid, 'targetOrg')
    export(func_alumni_info(target_org_uuid), target_org_uuid, 'alumniInfo')
    export(func_subsequent_orgs_info(target_org_uuid), target_org_uuid, 'subsequentOrgsInfo')
    export(func_relations(target_org_uuid), target_org_uuid, 'relations')

96ab87ca-00b5-2ebc-f218-86262954e320
a367b036-5952-5435-7541-ad7ee8869e24
df662812-7f97-0b43-9d3e-12f64f504fbb
022417b5-4980-6c54-0f3c-6736bbbb1a5e
2cc3a5de-2303-aa00-cd1a-50bd96420392
34035c51-8f16-4836-8f02-103392479a92
7b74b50c-4468-b7ec-2ff9-bf3286a399c9
d39bb6dc-582d-4878-a224-005497e03766
439d3478-40fa-e6bc-9b71-f1bfa8296f52
6093de34-5382-f34c-a207-efa92470048b
fd80725f-53fc-7009-9878-aeecf1e9ffbb
6acfa7da-1dbd-936e-d985-cf07a1b27711
eef9eab2-4c50-f0a3-12b8-ce721fa2cc81
cc68526b-b2d7-4f7f-cfa7-d93b23716027
2c23e7ad-4441-5320-fdf9-5aa7007fec0b
b3a7efa7-e3d3-2d4c-962c-b9fdb1da496f
1eb37109-3b93-01a9-177f-fee2cb1bfcdc
9a0e860a-7743-28e2-05c0-2b08646d0fe1
988b8953-a085-8de4-babb-fae5bb895761
643d60b4-bfa8-ee61-3316-5af0d8325f33
0d5171b3-68b3-37c3-cb50-8cd8ccb8930b
4b724583-6013-30ca-88b0-35e5c6e71ad5
297df5af-da41-a709-7df2-7e7f04e53454
e56b0ceb-bb30-bbec-805e-d5dc7412dcb1
a6b663d3-586a-ad04-924c-1ca87763b2fb
08639f0b-56fd-997f-5c9d-0ca3b5d9672b
1966bb69-4bc9-1077-c181-fe3d67509160
8

In [29]:
target_org_uuid = 'ada0db7e-2854-4d21-8e6a-eee86e978182'

export(func_target_org(target_org_uuid), target_org_uuid, 'targetOrg')
export(func_alumni_info(target_org_uuid), target_org_uuid, 'alumniInfo')
export(func_subsequent_orgs_info(target_org_uuid), target_org_uuid, 'subsequentOrgsInfo')
export(func_relations(target_org_uuid), target_org_uuid, 'relations')

## Full options list export (one file for DB)

In [30]:
values = [company["value"] for company in json.loads(options_json)]

In [31]:
target_org = organisations.loc[organisations['org_uuid'].isin(values)][['org_uuid', 'org_name', 'org_country_code', 'org_region', 'org_city', 'short_description', 'total_funding_usd', 'founded_on', 'org_logo_url']]

# Add exit info
target_org = pd.merge(target_org, acquisitons_ipos, how = 'left', on = 'org_uuid')

result = target_org.rename(columns={
    'org_uuid': 'orgUuid',
    'org_name': 'orgName',
    'org_country_code': 'orgCountryCode',
    'org_region': 'orgRegion',
    'org_city': 'orgCity',
    'short_description': 'shortDescription',
    'total_funding_usd': 'totalFundingUsd',
    'founded_on': 'foundedOn',
    'org_logo_url': 'orgLogoUrl',
    'exit_type': 'exitType',
    'exit_date': 'exitDate',
    'exit_valuation': 'exitValuation',
    'acquirer_uuid': 'acquirerUuid',
    'acquirer_name': 'acquirerName',
    'acquisition_type': 'acquisitionType',
}).drop_duplicates()

result.to_csv('data/output/' + 'targetOrgs' + '_' + 'full' + '.csv', index=False)

In [32]:
columns = [
    'record_uuid',
    'person_uuid',
    'org_uuid_target',
    'started_on_target',
    'ended_on_target',
    'job_title_target',
    'job_type_target',
    'org_name_target',
    'org_country_code_target',
    'org_city_target',
    'founded_on_target',
    'person_name',
    'person_logo_url',
    'org_uuid_subsequent',
    'started_on_subsequent',
    'ended_on_subsequent',
    'job_title_subsequent',
    'relation_type',
    'org_name_subsequent',
    'org_country_code_subsequent',
    'org_city_subsequent',
    'founded_on_subsequent',
    'short_description_subsequent',
    'total_funding_usd_subsequent',
    'org_logo_url_subsequent',
    'acquirer_uuid_subsequent',
    'exit_type_subsequent',
    'acquirer_name_subsequent',
    'exit_date_subsequent',
    'acquisition_type_subsequent',
    'exit_valuation_subsequent'
]

alumni_master = alumni_master.loc[alumni_master['org_uuid_target'].isin(values)][columns]

result = alumni_master.rename(columns={
    'record_uuid': 'recordUuid',
    'person_uuid': 'personUuid',
    'org_uuid_target': 'orgUuidTarget',
    'started_on_target': 'startedOnTarget',
    'ended_on_target': 'endedOnTarget',
    'job_title_target': 'jobTitleTarget',
    'job_type_target': 'jobTypeTarget',
    'org_name_target': 'orgNameTarget',
    'org_country_code_target': 'orgCountryCodeTarget',
    'org_city_target': 'orgCityTarget',
    'founded_on_target': 'foundedOnTarget',
    'person_name': 'personName',
    'person_logo_url': 'personLogoUrl',
    'org_uuid_subsequent': 'orgUuidSubsequent',
    'started_on_subsequent': 'startedOnSubsequent',
    'ended_on_subsequent': 'endedOnSubsequent',
    'job_title_subsequent': 'jobTitleSubsequent',
    'relation_type': 'relationType',
    'org_name_subsequent': 'orgNameSubsequent',
    'org_country_code_subsequent': 'orgCountryCodeSubsequent',
    'org_city_subsequent': 'orgCitySubsequent',
    'founded_on_subsequent': 'foundedOnSubsequent',
    'short_description_subsequent': 'shortDescriptionSubsequent',
    'total_funding_usd_subsequent': 'totalFundingUsdSubsequent',
    'org_logo_url_subsequent': 'orgLogoUrlSubsequent',
    'acquirer_uuid_subsequent': 'acquirerUuidSubsequent',
    'exit_type_subsequent': 'exitTypeSubsequent',
    'acquirer_name_subsequent': 'acquirerNameSubsequent',
    'exit_date_subsequent': 'exitDateSubsequent',
    'acquisition_type_subsequent': 'acquisitionTypeSubsequent',
    'exit_valuation_subsequent': 'exitValuationSubsequent'
}).drop_duplicates()

alumni_master.to_csv('data/output/' + 'alumni_master' + '_' + 'full' + '.csv', index=False)